In [ ]:
# Notebook: nb_utils_config (Python engine)
# Purpose: Shared configuration + utility helpers for __PROJECT_DISPLAY_NAME__
# Engine: python (single-node polars / duckdb / delta-rs)
# Usage: %run nb_utils_config   (bare notebook-item name, not a repo path; works in Python notebooks for notebook items)
#
# No Spark session here: no `spark`, no `pyspark.sql.functions`, no `F.`.
# Delta I/O goes through delta-rs (`deltalake`) + polars; reads/writes resolve
# through `table_path()` so the "Unidentified table" registration gotcha is
# handled in exactly one place.


In [ ]:
import os
from datetime import datetime, timezone

import polars as pl
import deltalake
from deltalake import write_deltalake, DeltaTable
import notebookutils


# --- delta-rs write compatibility (owns the schema_mode/engine gotcha in ONE place) ---
# delta-rs < 0.18 defaults to the pyarrow writer, which REJECTS schema_mode:
#   ValueError: schema_mode 'merge' is not supported in pyarrow engine. Use engine=rust
# So pass engine="rust" there. delta-rs >= 0.18 made rust the only writer and later
# REMOVED the engine kwarg entirely, so we must pass NOTHING on those runtimes
# (else TypeError: unexpected keyword 'engine'). Spread into every write:
#   write_deltalake(table_path(...), arrow, mode=..., schema_mode=..., **DELTA_WRITE_KWARGS)
def _delta_write_kwargs() -> dict:
    try:
        major, minor = (int(p) for p in deltalake.__version__.split(".")[:2])
    except Exception:
        return {}
    return {"engine": "rust"} if (major, minor) < (0, 18) else {}


DELTA_WRITE_KWARGS = _delta_write_kwargs()


In [ ]:
# --- Project Configuration ---
PROJECT_NAME = "__PROJECT_DISPLAY_NAME__"
WORKSPACE = "__WORKSPACE__"

# Lakehouse names
BRONZE_LAKEHOUSE = "__BRONZE_LAKEHOUSE__"
SILVER_LAKEHOUSE = "__SILVER_LAKEHOUSE__"
GOLD_LAKEHOUSE = "__GOLD_LAKEHOUSE__"

# Schema-enabled lakehouse? Drives table_path() resolution.
# Source of truth: project-config.yml (medallion.layers.*.schema_enabled).
# Default False (classic lakehouse) -- the safe default; a classic path under a
# classic lakehouse always registers. Flip to True for schema-enabled lakehouses
# so writes land under Tables/dbo/<name> and avoid the "Unidentified" folder.
SCHEMA_ENABLED = False


In [ ]:
# --- Table Path Resolver (owns the registration gotcha) ---
def table_path(name: str, schema_enabled: bool = None) -> str:
    """Resolve a managed Delta table path under the default lakehouse mount.

    schema-enabled lakehouse -> /lakehouse/default/Tables/dbo/<name>
    classic lakehouse        -> /lakehouse/default/Tables/<name>

    Never hard-code Tables/...; always go through here so the schema-enabled vs
    classic "Unidentified table" registration gotcha is handled in one place.
    """
    if schema_enabled is None:
        schema_enabled = SCHEMA_ENABLED
    base = "/lakehouse/default/Tables"
    return f"{base}/dbo/{name}" if schema_enabled else f"{base}/{name}"


# --- Table Naming Functions ---
def bronze_table(source_name: str) -> str:
    """Generate bronze table name."""
    return f"bronze_{source_name}"

def silver_table(entity_name: str) -> str:
    """Generate silver table name."""
    return f"silver_{entity_name}"

def dim_table(entity_name: str) -> str:
    """Generate dimension table name."""
    return f"dim_{entity_name}"

def fct_table(process_name: str) -> str:
    """Generate fact table name."""
    return f"fct_{process_name}"


In [ ]:
# --- Read Helpers (delta tables are OneLake-backed; mount-unaffected) ---
def read_bronze(source_name: str) -> "pl.DataFrame":
    """Read a bronze Delta table by source name as a polars DataFrame.

    Silver notebooks should ONLY read upstream data via read_bronze(...) --
    never pl.read_csv / read_parquet / scan external paths. This keeps the
    silver "bronze-only" contract intact across engines.
    """
    return pl.read_delta(table_path(bronze_table(source_name)))


In [ ]:
# --- Metadata Column Functions ---
def add_bronze_metadata(df: "pl.DataFrame", source_file: str = None) -> "pl.DataFrame":
    """Add standard bronze metadata columns to a polars DataFrame.

    _load_timestamp : UTC load time as a literal column (no per-row UDF needed)
    _source_file    : source path literal (single-node has no input_file_name())
    _load_id        : Fabric run id (same idiom as PySpark)
    """
    load_id = notebookutils.runtime.context.get("currentRunId", "manual")
    return df.with_columns(
        pl.lit(datetime.now(timezone.utc)).alias("_load_timestamp"),
        pl.lit(source_file if source_file is not None else "").alias("_source_file"),
        pl.lit(load_id).alias("_load_id"),
    )


def add_silver_metadata(df: "pl.DataFrame") -> "pl.DataFrame":
    """Swap bronze metadata for silver metadata.

    Drops the bronze ingestion columns (_load_timestamp/_source_file/_load_id)
    if present and stamps a silver processing timestamp.
    """
    drop_cols = [c for c in ("_load_timestamp", "_source_file", "_load_id") if c in df.columns]
    if drop_cols:
        df = df.drop(drop_cols)
    return df.with_columns(
        pl.lit(datetime.now(timezone.utc)).alias("_silver_processed_timestamp"),
    )


In [ ]:
# --- Validation Functions (delta-rs; do not materialize to count) ---
def validate_row_count(table_name: str, min_rows: int = 1) -> int:
    """Validate a Delta table has at least min_rows, counting cheaply via delta-rs."""
    count = DeltaTable(table_path(table_name)).to_pyarrow_dataset().count_rows()
    assert count >= min_rows, f"FAIL: {table_name} has {count} rows (min: {min_rows})"
    print(f"PASS: {table_name} has {count} rows")
    return count


def validate_no_nulls(table_name: str, columns: list) -> None:
    """Validate that specified columns have no null values."""
    df = pl.read_delta(table_path(table_name))
    for col_name in columns:
        null_count = df.select(pl.col(col_name).is_null().sum()).item()
        assert null_count == 0, f"FAIL: {col_name} in {table_name} has {null_count} nulls"
        print(f"PASS: {col_name} has 0 nulls")


def validate_unique(table_name: str, columns: list) -> None:
    """Validate that specified columns form a unique key."""
    df = pl.read_delta(table_path(table_name))
    total = df.height
    distinct = df.select(columns).unique().height
    dups = total - distinct
    assert dups == 0, f"FAIL: {dups} duplicate rows on {columns} in {table_name}"
    print(f"PASS: {columns} unique in {table_name} ({total} rows)")


In [ ]:
print(f"Configuration loaded for: {PROJECT_NAME}")
print(f"Workspace: {WORKSPACE}")
print(f"Lakehouses: {BRONZE_LAKEHOUSE} / {SILVER_LAKEHOUSE} / {GOLD_LAKEHOUSE}")
print(f"Schema-enabled lakehouse: {SCHEMA_ENABLED}")
